# Autonomous Trailer: Knowledge-Based Route Agent

ICS 3202 - Practical Exercise 4

This notebook models two Nairobi-to-Kisumu candidate corridors using directed road-distance facts and adds fuel-stop facts from OpenStreetMap Overpass.

## 1. Task and scope

The agent searches the directed road graph from Nairobi to Kisumu using Uniform Cost Search (UCS), with road distance in kilometres as the path cost. The six segment distances below were supplied with Google Maps direction links.

### Evidence and assumptions

- Segment distances sum to 350 km via Nakuru and Kericho, and 355 km via Narok and Bomet.
- Only the Nairobi-to-Kisumu directions supplied are encoded. Reverse arcs are not assumed.
- The search minimizes distance only. It does not use live traffic or verified road-condition data. Fuel-stop records are fetched at notebook runtime and may be incomplete or stale.

## 2. Assemble knowledge and data sources

The directed road facts and distances come from the six Google Maps segment links supplied for the two candidate corridors. The notebook calculates the totals from those segments and runs UCS over the resulting graph.

Fuel stations are retrieved from OpenStreetMap using Overpass. The query searches within 5 km of each modeled town centre and records each result's OSM ID, name, coordinates, and tags. This point-of-interest lookup does not verify opening hours, fuel availability, trailer access, or safety.

The AllanVikiru notebook link supplied in the instructions is used as a workflow-format reference only; its traffic-light example is not route data.

## 3. Vocabulary

- Location(name, region, latitude, longitude): a modeled town.
- Road(from, to, corridor): a directed traversable connection.
- Distance(from, to, km): the non-negative distance cost for a directed road.
- FuelStop(osm_id, name, latitude, longitude, tags): an OpenStreetMap fuel-station point returned by the API.
- Blocked(from, to): an optional scenario fact that excludes a directed road from search.

The route cost is the sum of directed road distances. UCS expands the frontier node with the smallest accumulated distance and returns a minimum-distance path for the encoded graph.

## 4. Encode the starting facts

The six directed road distances below are transcribed from the Google Maps direction links provided with the route data. Only the Nairobi-to-Kisumu direction is encoded. The route totals are calculated from the segment facts.

In [ ]:
from datetime import datetime, timezone

locations = {
    "Nairobi": {"region": "Central Highlands", "lat": -1.2864, "lon": 36.8172},
    "Nakuru":  {"region": "Rift Valley", "lat": -0.3031, "lon": 36.0800},
    "Kericho": {"region": "Rift Valley", "lat": -0.3676, "lon": 35.2831},
    "Narok":   {"region": "Rift Valley", "lat": -1.0833, "lon": 35.8667},
    "Bomet":   {"region": "Rift Valley", "lat": -0.7813, "lon": 35.3416},
    "Kisumu":  {"region": "Lake Victoria Basin", "lat": -0.0917, "lon": 34.7680},
}

# Directed distance facts for the Nairobi-to-Kisumu task.
# The source links and reported values are listed in the Sources section.
roads = [
    {"from": "Nairobi", "to": "Nakuru", "km": 159,
     "corridor": "Via Nakuru and Kericho"},
    {"from": "Nakuru", "to": "Kericho", "km": 107,
     "corridor": "Via Nakuru and Kericho"},
    {"from": "Kericho", "to": "Kisumu", "km": 84,
     "corridor": "Via Nakuru and Kericho"},
    {"from": "Nairobi", "to": "Narok", "km": 143,
     "corridor": "Via Narok and Bomet"},
    {"from": "Narok", "to": "Bomet", "km": 81,
     "corridor": "Via Narok and Bomet"},
    {"from": "Bomet", "to": "Kisumu", "km": 131,
     "corridor": "Via Narok and Bomet"},
]

route_totals = {}
for edge in roads:
    route_totals[edge["corridor"]] = route_totals.get(edge["corridor"], 0) + edge["km"]

print(f"Locations: {len(locations)}; directed road facts: {len(roads)}")
for route_name, total in route_totals.items():
    print(f"{route_name}: {total} km")

## 5. Add fuel-stop facts from an API

Overpass API returns current OpenStreetMap objects matching amenity=fuel within 5 km of the town search centres. The result is time-sensitive; rerun to refresh it and keep the printed retrieval timestamp with the submitted output. If the public service is temporarily unavailable, the route facts still work.

In [ ]:
import json
import urllib.parse
import urllib.request

OVERPASS_URL = "https://overpass-api.de/api/interpreter"

def fetch_fuel_stops(locations, radius_m=5000, timeout_s=45):
    selectors = []
    for place, info in locations.items():
        selectors.append(
            f'nwr["amenity"="fuel"](around:{radius_m},{info["lat"]},{info["lon"]});'
        )
    query = "[out:json][timeout:25];(\n" + "\n".join(selectors) + "\n);out center tags;"
    payload = urllib.parse.urlencode({"data": query}).encode("utf-8")
    request = urllib.request.Request(
        OVERPASS_URL, data=payload, method="POST",
        headers={"User-Agent": "ICS3202-AutonomousTrailer-StudentProject/1.0"}
    )
    with urllib.request.urlopen(request, timeout=timeout_s) as response:
        data = json.loads(response.read().decode("utf-8"))

    stops, seen = [], set()
    for element in data.get("elements", []):
        key = (element.get("type"), element.get("id"))
        if key in seen:
            continue
        seen.add(key)
        tags = element.get("tags", {})
        center = element.get("center", {})
        stops.append({
            "osm_type": element.get("type"), "osm_id": element.get("id"),
            "name": tags.get("name", "Unnamed fuel stop"),
            "brand": tags.get("brand"),
            "latitude": element.get("lat", center.get("lat")),
            "longitude": element.get("lon", center.get("lon")),
            "tags": tags,
        })
    return stops

retrieved_at_utc = datetime.now(timezone.utc).isoformat(timespec="seconds")
try:
    fuel_stops = fetch_fuel_stops(locations)
    print(f"Retrieved {len(fuel_stops)} OSM fuel-stop facts at {retrieved_at_utc} UTC")
    for stop in fuel_stops[:10]:
        print(stop["name"], stop["latitude"], stop["longitude"])
except Exception as exc:
    fuel_stops = []
    print("Fuel-stop lookup failed; no live fuel facts were added.")
    print(type(exc).__name__, str(exc))

## 6. Encode the problem instance and run UCS

The search tree below records each candidate edge generated from an expanded node, its cumulative distance, and whether it improves the known cost to that town.

In [ ]:
from heapq import heappush, heappop
from math import inf


def uniform_cost_route(start, goal, blocked_edges=None):
    if start not in locations or goal not in locations:
        return None
    blocked_edges = set(blocked_edges or ())
    adjacency = {place: [] for place in locations}
    for edge in roads:
        if (edge["from"], edge["to"]) not in blocked_edges:
            adjacency[edge["from"]].append(edge)

    frontier = [(0, start)]
    best_cost = {start: 0}
    parent = {}
    search_tree = []
    expansion_order = []

    while frontier:
        cost, here = heappop(frontier)
        if cost != best_cost.get(here):
            continue
        expansion_order.append({"town": here, "cumulative_km": cost})
        if here == goal:
            path = [goal]
            while path[-1] != start:
                path.append(parent[path[-1]])
            path.reverse()
            return {
                "path": path,
                "distance_km": cost,
                "search_tree": search_tree,
                "expansion_order": expansion_order,
            }

        for edge in adjacency[here]:
            next_cost = cost + edge["km"]
            child = edge["to"]
            accepted = next_cost < best_cost.get(child, inf)
            search_tree.append({
                "parent": here, "child": child,
                "edge_km": edge["km"], "cumulative_km": next_cost,
                "kept_in_frontier": accepted,
            })
            if accepted:
                best_cost[child] = next_cost
                parent[child] = here
                heappush(frontier, (next_cost, child))
    return None

route = uniform_cost_route("Nairobi", "Kisumu")
print("UCS route query: Nairobi to Kisumu")
print({"path": route["path"], "distance_km": route["distance_km"]})
print("\nSearch-tree candidate edges:")
for branch in route["search_tree"]:
    print(branch)
print("\nExpanded in cumulative-cost order:")
for item in route["expansion_order"]:
    print(item)

## 7. Explain the answer from the knowledge base

Each selected hop corresponds to a directed Road and Distance fact. UCS expands towns in increasing accumulated distance; for this graph it selects the 350 km route via Nakuru and Kericho.

In [ ]:
def explain_route(result, blocked_edges=None):
    if result is None:
        return "No known unblocked route. Check the graph facts."
    blocked_edges = set(blocked_edges or ())
    segments = []
    for a, b in zip(result["path"], result["path"][1:]):
        edge = next(e for e in roads if e["from"] == a and e["to"] == b)
        segments.append(f"{a} -> {b}: {edge['km']} km via {edge['corridor']}")
    return "\n".join(segments) + f"\nTotal: {result['distance_km']} km"

print(explain_route(route))

## 8. Query the knowledge base under a road closure scenario

The closure below is hypothetical only. It demonstrates how a blocked-road fact changes the result; it is not a report of current road conditions.

In [ ]:
blocked_edges = {("Nairobi", "Nakuru")}
closure_scenario = uniform_cost_route("Nairobi", "Kisumu", blocked_edges)
print("Hypothetical closure of Nairobi -> Nakuru:")
print(explain_route(closure_scenario, blocked_edges))

print("\nKnown fuel-stop records (first five):")
for stop in fuel_stops[:5]:
    print({k: stop[k] for k in ("name", "brand", "latitude", "longitude")})

## 9. Debugging and edge-case queries

Inspect behaviour for an unknown town, the normal route, and a scenario that blocks both outgoing roads from Nairobi.

In [ ]:
print("Unknown start:", uniform_cost_route("UnknownTown", "Kisumu"))
print("Normal route:", uniform_cost_route("Nairobi", "Kisumu")["path"])
print(
    "No-route scenario:",
    uniform_cost_route(
        "Nairobi", "Kisumu",
        {("Nairobi", "Nakuru"), ("Nairobi", "Narok")}
    )
)

## 10. Reflection and limitations

This knowledge base separates map facts from route-search rules. UCS can explain its distance-based choice, and the hypothetical blocked-road query shows how a road-status fact can alter the result. The six Google Maps segment values sum to 350 km via Nakuru and Kericho and 355 km via Narok and Bomet. The graph models only these two candidate paths and only the Nairobi-to-Kisumu directions supplied.

The fuel-stop response is an OpenStreetMap snapshot: it may miss stations or contain outdated records, and it does not confirm opening hours, fuel availability, truck access, or safety. The system has no live traffic feed and should not be described as traffic-aware.

## Group-work evidence

Add the group's authentic meeting minutes or a group photo here before submission. Do not leave this placeholder in the submitted notebook.

## Sources and attribution

- Google Maps route distance links and segment values supplied for this notebook:
  - Nairobi -> Nakuru: 159 km - https://www.google.com/maps/dir/Nairobi,+Kenya/Nakuru,+Kenya/data=!4m8!4m7!1m2!1m1!19sChIJp0lN2HIRLxgRTJKXslQCz_c!1m2!1m1!19sChIJ7wthz5CNKRgRGsJ8vDMY8vI!3e0
  - Nakuru -> Kericho: 107 km - https://www.google.com/maps/dir/Nakuru,+Kenya/Kericho,+Kenya/data=!4m8!4m7!1m2!1m1!19sChIJ7wthz5CNKRgRGsJ8vDMY8vI!1m2!1m1!19sChIJGQ4Kp3hXKhgRlziS5gASEZ4!3e0
  - Kericho -> Kisumu: 84 km - https://www.google.com/maps/dir/Kericho,+Kenya/Kisumu,+Kenya/data=!4m8!4m7!1m2!1m1!19sChIJGQ4Kp3hXKhgRlziS5gASEZ4!1m2!1m1!19sChIJHchKrTekKhgR0o0k1jmkEiA!3e0
  - Nairobi -> Narok: 143 km - https://www.google.com/maps/dir/Nairobi,+Kenya/Narok,+Kenya/data=!4m8!4m7!1m2!1m1!19sChIJp0lN2HIRLxgRTJKXslQCz_c!1m2!1m1!19sChIJRQaL3gEDLBgROY78WUSsbCI!3e0
  - Narok -> Bomet: 81 km - https://www.google.com/maps/dir/Narok,+Kenya/Bomet,+Kenya/data=!4m8!4m7!1m2!1m1!19sChIJRQaL3gEDLBgROY78WUSsbCI!1m2!1m1!19sChIJ75DqwoGZKxgR6-_srbaV1XQ!3e0
  - Bomet -> Kisumu: 131 km - https://www.google.com/maps/dir/Bomet,+Kenya/Kisumu,+Kenya/data=!4m8!4m7!1m2!1m1!19sChIJ75DqwoGZKxgR6-_srbaV1XQ!1m2!1m1!19sChIJHchKrTekKhgR0o0k1jmkEiA!3e0
- [Overpass API](https://wiki.openstreetmap.org/wiki/Overpass_API) and [Overpass QL guide](https://wiki.openstreetmap.org/wiki/Overpass_API/Language_Guide).
- [OSRM project documentation](https://github.com/Project-OSRM/osrm-backend) describes routing services over OpenStreetMap data; OSRM is not used as a live traffic source here.
- Fuel-stop data are from [OpenStreetMap contributors](https://www.openstreetmap.org/copyright), available under the Open Database License (ODbL).
- Workflow-format reference supplied in the instructions: https://github.com/AllanVikiru/ArtificialIntelligence/blob/kba/Knowledge_Engineering.ipynb